In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import joblib
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
np.random.seed(42)

: 

In [5]:
# Hämta MNIST-datasetet
mnist = fetch_openml('mnist_784', version=1, cache=True, as_frame=False)

# Definiera X och y
X = mnist["data"]
y = mnist["target"].astype(np.uint8)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Spara data och mål som NumPy-filer
np.save("X_mnist.npy",X)
np.save("y_mnist.npy", y)

# Kontrollera om några bilder är tomma (alla pixelvärden = 0)
empty_indices = np.where(X.sum(axis=1) == 0)[0]

# Skriv ut information om de tomma bilderna
print(f"Antal tomma bilder: {len(empty_indices)}")
print(f"Index för tomma bilder: {empty_indices}")

X shape: (70000, 784)
y shape: (70000,)
Antal tomma bilder: 0
Index för tomma bilder: []


In [6]:
# Ladda sparade NumPy-filer
def load_mnist_data():
    X = np.load("X_mnist.npy")  # Pixeldata
    y = np.load("y_mnist.npy").astype(np.uint8)  # Etiketter som uint8
    print(f"Data laddad: X shape = {X.shape}, y shape = {y.shape}")
    return X, y  # Returnera data och mål

# Använd funktionen för att ladda data
X, y = load_mnist_data()

Data laddad: X shape = (70000, 784), y shape = (70000,)


In [9]:
from sklearn.model_selection import train_test_split

# Fördela data i tränings- och testset så att testdatan är 35% av den totala datan
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.4285, random_state=42)

# Ange antalet träningsobservationer som ska vara 25 000
train_size = 25000

# Fördela tränings- och valideringsdatan så att valideringsdatan utgör resterande del
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, train_size=train_size, random_state=42)

# Kontrollera storleken på tränings- och valideringsdatan
print(f'Train data shape: {X_train.shape}, {y_train.shape}')
print(f'Validation data shape: {X_val.shape}, {y_val.shape}')
print(f'Test data shape: {X_test.shape}, {y_test.shape}')

Train data shape: (25000, 784), (25000,)
Validation data shape: (15005, 784), (15005,)
Test data shape: (29995, 784), (29995,)


In [6]:
negative_values = X_train[X_train < 0]
print(negative_values)

[]


In [7]:
print("Finns NaN-värden i X_train?", np.isnan(X_train).any())
print("Finns oändliga värden i X_train?", np.isinf(X_train).any())

Finns NaN-värden i X_train? False
Finns oändliga värden i X_train? False


In [ ]:
some_digit = X[0]
some_digit_image = some_digit.reshape(28, 28)
plt.imshow(some_digit_image, cmap="binary")
plt.axis("off")
plt.show()

In [ ]:
y[0]

In [ ]:
sns.countplot(x=y)
plt.title('Distribution of Labels')
plt.show()

In [ ]:
# Räkna antalet förekomster av varje siffra och visualisera distributionen
label_counts = pd.Series(y).value_counts().sort_index()
print(label_counts)

# Skapa en barplot över fördelningen
plt.figure(figsize=(10, 6))
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.xlabel('Siffror')
plt.ylabel('Antal bilder')
plt.title('Fördelning av siffror i MNIST-datasetet')
plt.show()

In [ ]:
#Skapa DataFrame för analys
df_train = pd.DataFrame(X_train)
print(df_train.describe())
print(df_train.info())
print(df_train.head())

In [ ]:
# Skapa ett histogram för varje siffra
fig, axes = plt.subplots(2, 5, figsize=(20, 10))  # Skapar en 2x5 layout för subplots

# Loopar igenom varje siffra (0-9)
for i in range(10):
    ax = axes[i // 5, i % 5]  # Bestämmer positionen för subplot
    data = X_train[y_train == i].flatten()  # Filtrerar data för den aktuella siffran
    sns.histplot(data, bins=30, kde=True, ax=ax)
    ax.set_title(f'Siffra {i}')
    ax.set_xlabel('Värde')
    ax.set_ylabel('Frekvens')

plt.tight_layout()
plt.show()

Initering och träning av modell logistik regression, resultat sparas och kan återhämtas.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Skapa en pipeline med KNN
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Standardisera data
    ('model', KNeighborsClassifier())
])

# Definiera parameterområdet för grid search
param_grid = {
    'model__n_neighbors': [3, 5, 7],
    'model__weights': ['uniform', 'distance', None],
    'model__metric': ['euclidean', 'manhattan']
}

# Initiera GridSearchCV med pipelinen och parameterområdet
grid_search = GridSearchCV(knn_pipeline, param_grid, cv=5, scoring='accuracy', refit=True)
grid_search.fit(X_train, y_train)

# Hämta bästa parametrarna och resultatet
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Spara den bästa pipelinen
best_pipeline = grid_search.best_estimator_
joblib.dump(best_pipeline, 'optimized_knn.pkl')

print("Bästa parametrarna:", best_params)
print("Bästa valideringsnoggrannheten:", best_score)

Bästa parametrarna: {'model__metric': 'manhattan', 'model__n_neighbors': 3, 'model__weights': 'distance'}
Bästa valideringsnoggrannheten: 0.94848
